In [ ]:
# =============================================================================
# CÉLULA 0: PARÂMETROS PARA PAPERMILL
# =============================================================================
EMAIL_REMETENTE = "ab11.94958191@gmail.com"
SENHA_APP = ""


In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO DE BIBLIOTECAS E IMPORTAÇÕES
# =============================================================================
!pip install yfinance pandas-ta optuna feedparser vaderSentiment googletrans==4.0.0rc1 --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
import time
import warnings
warnings.filterwarnings("ignore")

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import feedparser
from googletrans import Translator
import optuna

print("✅ Bibliotecas instaladas e carregadas.")


In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (COMPLETAS, CORRIGIDAS, LOGARÍTMICAS)
# =============================================================================

# -----------------------------------------------------------------------------
# FUNÇÕES TÉCNICAS BÁSICAS (MANTIDAS COMO ESTAVAM)
# -----------------------------------------------------------------------------
def calcular_eficiencia_candle(df):
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low']
    range_total = range_total.replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta = df['Close'] > df['Open']
    baixa = df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df, janela=20):
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
    df_temp['adx'] = adx['ADX_14']
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty:
        return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33:
            return 0  # Lateral
        elif v > v_p67 or a > a_p67:   # CORRIGIDO
            return 2  # Volátil/Tendência Forte
        else:
            return 1  # Tendência Suave
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

# PIVÔS (JÁ CORRIGIDOS PARA INCLUIR CANDLES RECENTES)
def detectar_swing_low(df, janela=10):
    lows = df['Low'].values
    swing_lows = []
    for i in range(janela, len(lows)):
        if i - janela >= 0:
            lookback = lows[i-janela:i]
            if lows[i] <= min(lookback):
                swing_lows.append(lows[i])
    if swing_lows:
        return float(swing_lows[-1])
    else:
        return float(df['Low'].min())

def detectar_swing_high(df, janela=10):
    highs = df['High'].values
    swing_highs = []
    for i in range(janela, len(highs)):
        if i - janela >= 0:
            lookback = highs[i-janela:i]
            if highs[i] >= max(lookback):
                swing_highs.append(highs[i])
    if swing_highs:
        return float(swing_highs[-1])
    else:
        return float(df['High'].max())

# -----------------------------------------------------------------------------
# LTA / LTB EM ESCALA LOGARÍTMICA (CORREÇÃO MATEMÁTICA)
# -----------------------------------------------------------------------------
def calcular_lta_pivos(df, janela_pivo=5):
    lows = df['Low'].values
    # Trabalhamos com logaritmos para capturar a verdadeira tendência percentual
    log_lows = np.log(lows)
    fundos = []
    for i in range(janela_pivo, len(log_lows) - janela_pivo):
        if log_lows[i] == min(log_lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, log_lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        x_atual = len(log_lows) - 1
        inclinacao = (y2 - y1) / (x2 - x1)
        suporte_log = y2 + inclinacao * (x_atual - x2)
        return np.exp(suporte_log)  # retorna preço real
    else:
        return None

def calcular_ltb_pivos(df, janela_pivo=5):
    highs = df['High'].values
    log_highs = np.log(highs)
    topos = []
    for i in range(janela_pivo, len(log_highs) - janela_pivo):
        if log_highs[i] == max(log_highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, log_highs[i]))
    if len(topos) >= 2:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        if y1 > y2:  # garante inclinação negativa
            x_atual = len(log_highs) - 1
            inclinacao = (y2 - y1) / (x2 - x1)
            resistencia_log = y2 + inclinacao * (x_atual - x2)
            return np.exp(resistencia_log)
    return None

# PADRÕES DE CANDLE (INALTERADOS)
def detectar_padrao_altista(df_diario):
    if len(df_diario) < 3:
        return False
    # (corpo da função permanece idêntico)
    ultimo = df_diario.iloc[-1]
    penultimo = df_diario.iloc[-2]
    corpo_ult = abs(ultimo['Close'] - ultimo['Open'])
    range_ult = ultimo['High'] - ultimo['Low']
    sombra_inf_ult = min(ultimo['Close'], ultimo['Open']) - ultimo['Low']
    sombra_sup_ult = ultimo['High'] - max(ultimo['Close'], ultimo['Open'])
    if range_ult > 0:
        if sombra_inf_ult >= 2 * corpo_ult and sombra_sup_ult <= 0.3 * corpo_ult:
            return True
    corpo_pen = abs(penultimo['Close'] - penultimo['Open'])
    if penultimo['Close'] < penultimo['Open'] and ultimo['Close'] > ultimo['Open']:
        if ultimo['Open'] <= penultimo['Close'] and ultimo['Close'] >= penultimo['Open']:
            return True
    if penultimo['Close'] < penultimo['Open'] and ultimo['Close'] > ultimo['Open']:
        meio_corpo_pen = (penultimo['Open'] + penultimo['Close']) / 2
        if ultimo['Open'] <= penultimo['Close'] and ultimo['Close'] >= meio_corpo_pen:
            return True
    return False

def detectar_padrao_baixista(df_diario):
    if len(df_diario) < 3:
        return False
    # (corpo da função permanece idêntico)
    ultimo = df_diario.iloc[-1]
    penultimo = df_diario.iloc[-2]
    corpo_ult = abs(ultimo['Close'] - ultimo['Open'])
    range_ult = ultimo['High'] - ultimo['Low']
    sombra_sup_ult = ultimo['High'] - max(ultimo['Close'], ultimo['Open'])
    sombra_inf_ult = min(ultimo['Close'], ultimo['Open']) - ultimo['Low']
    if range_ult > 0:
        if sombra_sup_ult >= 2 * corpo_ult and sombra_inf_ult <= 0.3 * corpo_ult:
            return True
    corpo_pen = abs(penultimo['Close'] - penultimo['Open'])
    if penultimo['Close'] > penultimo['Open'] and ultimo['Close'] < ultimo['Open']:
        if ultimo['Open'] >= penultimo['Close'] and ultimo['Close'] <= penultimo['Open']:
            return True
    if penultimo['Close'] > penultimo['Open'] and ultimo['Close'] < ultimo['Open']:
        meio_corpo_pen = (penultimo['Open'] + penultimo['Close']) / 2
        if ultimo['Open'] >= penultimo['Close'] and ultimo['Close'] <= meio_corpo_pen:
            return True
    return False

analyzer = SentimentIntensityAnalyzer()
translator = Translator()

def obter_sentimento_ticker(ticker):
    """Análise de sentimento com VADER + tradução português->inglês"""
    try:
        ticker_limpo = ticker.replace('.SA', '')
        url = f"https://news.google.com/rss/search?q={ticker_limpo}+B3&hl=pt-BR&gl=BR&ceid=BR:pt-419"
        feed = feedparser.parse(url)
        scores = []
        for entry in feed.entries[:3]:
            translated = translator.translate(entry.title, dest='en').text
            vs = analyzer.polarity_scores(translated)
            scores.append(vs['compound'])
        return np.mean(scores) if scores else 0.0
    except:
        return 0.0

def detectar_volume_anormal(df, periodo=20, limiar=1.5):
    if len(df) < periodo:
        return False
    vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
    vol_ultimo = df['Volume'].iloc[-1]
    return vol_ultimo >= vol_medio * limiar

def fractional_kelly(win_rate, payoff_ratio, frac=0.25):
    if payoff_ratio <= 0:
        return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    kelly = max(0.0, min(kelly, 0.25))
    return kelly * frac

# -----------------------------------------------------------------------------
# REFERÊNCIAS MACRO E FUNÇÕES DAS CAMADAS (INALTERADAS)
# -----------------------------------------------------------------------------
MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6),
    'PETR4.SA': ('CL=F', 0.8),
    'PETR3.SA': ('CL=F', 0.8),
    'CSNA3.SA': ('GC=F', 0.5),
    'GGBR4.SA': ('GC=F', 0.5),
    'CAML3.SA': ('WEAT', 0.4),
    'JBSS3.SA': ('WEAT', 0.5),
    'ABEV3.SA': ('CORN', 0.3),
    'RADL3.SA': ('XLP', 0.3),
    'PRIO3.SA': ('CL=F', 0.7),
}

def verificar_alinhamento_macro(ticker, direcao, cache_macro):
    """
    Retorna (alinhado: bool, score: int) para o alinhamento macro.
    Utiliza EMA 50 + inclinação (derivada) para capturar momento.
    """
    if ticker not in MACRO_REFERENCE:
        return True, 15
    ref, peso = MACRO_REFERENCE[ticker]
    if ref not in cache_macro or cache_macro[ref] is None:
        return True, 15
    precos = cache_macro[ref]
    if len(precos) < 55:
        return True, 15

    # EMA 50 (suavização exponencial)
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    ema50_atual = ema50.iloc[-1]
    ema50_lag4 = ema50.iloc[-5]  # valor defasado 4 semanas

    if pd.isna(ema50_atual) or pd.isna(ema50_lag4):
        return True, 15

    preco = precos[-1]
    inclinacao_positiva = ema50_atual > ema50_lag4
    inclinacao_negativa = ema50_atual < ema50_lag4

    if direcao == 'COMPRA' and preco > ema50_atual and inclinacao_positiva:
        return True, 30
    if direcao == 'VENDA' and preco < ema50_atual and inclinacao_negativa:
        return True, 30
    return False, 0

def avaliar_qualidade_volume(df_w, direcao):
    vol_ultimo = df_w['Volume'].iloc[-1]
    vol_medio = df_w['Volume'].rolling(20).mean().iloc[-1]
    eficiencia = df_w['Eficiencia'].iloc[-1] if 'Eficiencia' in df_w.columns else 0.5
    if pd.isna(vol_medio) or vol_medio == 0:
        return 'NEUTRO', 10
    if vol_ultimo > vol_medio * 1.5:
        if direcao == 'COMPRA' and eficiencia > 0.7:
            return 'ALTA_CONVICCAO', 25
        if direcao == 'VENDA' and eficiencia > 0.7:
            return 'ALTA_CONVICCAO', 25
        if direcao == 'COMPRA' and eficiencia < 0.5:
            return 'POSSIVEL_ARMADILHA', 15
        if direcao == 'VENDA' and eficiencia < 0.5:
            return 'POSSIVEL_ARMADILHA', 15
    return 'NEUTRO', 10

# -----------------------------------------------------------------------------
# WYCKOFF (INALTERADO)
# -----------------------------------------------------------------------------
def detectar_fase_wyckoff(df_w, suporte=None, resistencia=None):
    """
    Classifica a fase atual do ativo segundo os princípios de Wyckoff.
    Retorna: 'ACUMULACAO', 'DISTRIBUICAO' ou 'INDEFINIDO'.
    """
    if len(df_w) < 20:
        return 'INDEFINIDO'

    range_semanal = df_w['High'] - df_w['Low']
    range_medio_20 = range_semanal.rolling(20).mean()
    range_medio_4  = range_semanal.rolling(4).mean()
    estreitamento = range_medio_4.iloc[-1] < (range_medio_20.iloc[-1] * 0.7)
    if not estreitamento:
        return 'INDEFINIDO'

    preco_atual = df_w['Close'].iloc[-1]
    if suporte is None or resistencia is None:
        suporte = df_w['Low'].rolling(20).min().iloc[-1]
        resistencia = df_w['High'].rolling(20).max().iloc[-1]

    faixa = resistencia - suporte
    if faixa == 0:
        return 'INDEFINIDO'
    posicao_relativa = (preco_atual - suporte) / faixa

    candles = df_w.iloc[-8:]
    alta = candles['Close'] > candles['Open']
    baixa = candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0:
        return 'INDEFINIDO'
    razao_volume = vol_alta / vol_baixa

    if posicao_relativa < 0.4 and razao_volume > 1.2:
        return 'ACUMULACAO'
    if posicao_relativa > 0.6 and razao_volume < 0.8:
        return 'DISTRIBUICAO'
    return 'INDEFINIDO'

# -----------------------------------------------------------------------------
# FIBONACCI EM ESCALA LOGARÍTMICA (CORREÇÃO MATEMÁTICA)
# -----------------------------------------------------------------------------
def calcular_alvos_fibonacci(df, direcao):
    """
    Calcula alvos de expansão de Fibonacci usando amplitude logarítmica.
    Para compra: projeta a partir do último topo.
    Para venda: projeta a partir do último fundo.
    """
    if len(df) < 20:
        return {}
    
    try:
        swing_low = detectar_swing_low(df, janela=10)
        swing_high = detectar_swing_high(df, janela=10)
        
        if swing_low is None or swing_high is None or swing_low >= swing_high:
            return {}
        
        # Amplitude logarítmica (reflete a verdadeira variação percentual)
        amplitude_log = np.log(swing_high) - np.log(swing_low)
        
        # Projeções a partir do swing_high (compra) ou swing_low (venda)
        if direcao == 'COMPRA':
            base_log = np.log(swing_high)
            niveis = {
                '100%':   round(np.exp(base_log + amplitude_log * 1.000), 2),
                '161.8%': round(np.exp(base_log + amplitude_log * 1.618), 2),
                '261.8%': round(np.exp(base_log + amplitude_log * 2.618), 2),
                '423.6%': round(np.exp(base_log + amplitude_log * 4.236), 2),
            }
        else:  # VENDA
            base_log = np.log(swing_low)
            niveis = {
                '100%':   round(np.exp(base_log - amplitude_log * 1.000), 2),
                '161.8%': round(np.exp(base_log - amplitude_log * 1.618), 2),
                '261.8%': round(np.exp(base_log - amplitude_log * 2.618), 2),
                '423.6%': round(np.exp(base_log - amplitude_log * 4.236), 2),
            }
        return niveis
    except:
        return {}

def calcular_score_confianca(setup, alinhado_macro, qualidade_volume, score_volume, fase_wyckoff='INDEFINIDO'):
    score = 0
    score += 30 if alinhado_macro else 0
    score += score_volume
    if setup['Eficiência'] and setup['Eficiência'] > 0.8:
        score += 20
    elif setup['Eficiência'] and setup['Eficiência'] > 0.6:
        score += 10
    if setup['Sentimento'] and abs(setup['Sentimento']) > 0.3:
        score += 15
    if setup['Regime'] == 2:
        score += 10
    elif setup['Regime'] == 1:
        score += 5
    # BÔNUS WYCKOFF
    if setup['Direcao'] == 'COMPRA' and fase_wyckoff == 'ACUMULACAO':
        score += 20
    elif setup['Direcao'] == 'VENDA' and fase_wyckoff == 'DISTRIBUICAO':
        score += 20
    return score / 100.0

print("✅ Célula 2 carregada (Escala Logarítmica, Fibonacci Log, Pivôs Corrigidos).")


In [ ]:
# =============================================================================
# CÉLULA 3: FUNÇÃO PRINCIPAL BIDIRECIONAL
#       (utiliza as funções logarítmicas da Célula 2)
# =============================================================================

OTIMIZADO_SWING = {
    'atr_period': 14,
    'atr_mult': 1.8,
    'swing_window': 12,
    'lta_pivo_window': 6,
    'ltb_pivo_window': 6,
    'mm200_semanal': True,
    'mm200_diaria': True,
    'volume_limiar': 1.3
}

OTIMIZADO_POSITION = {
    'atr_period': 14,
    'atr_mult': 2.5,
    'swing_window': 24,
    'lta_pivo_window': 12,
    'ltb_pivo_window': 12,
    'mm50_mensal': True,
    'volume_limiar': 1.2
}

def analisar_swing_trade(ticker, df_w=None, df_d=None):
    try:
        if df_w is None:
            return None
        df_w = df_w.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_w.columns):
            rename_map = {'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}
            df_w.rename(columns={k:v for k,v in rename_map.items() if k in df_w.columns}, inplace=True)

        df_w.sort_index(inplace=True)
        if not isinstance(df_w.index, pd.DatetimeIndex):
            df_w.index = pd.to_datetime(df_w.index)

        df_w['Eficiencia'] = calcular_eficiencia_candle(df_w)
        df_w['Regime'] = detectar_regime(df_w)

        ultimo = df_w.iloc[-1]
        entrada = float(ultimo['Close'])
        if pd.isna(entrada) or entrada <= 0:
            return None

        recent_high = float(df_w['High'].rolling(window=min(52, len(df_w))).max().iloc[-1])
        recent_low  = float(df_w['Low'].rolling(window=min(52, len(df_w))).min().iloc[-1])

        reg_atual = int(ultimo['Regime']) if not pd.isna(ultimo['Regime']) else -1
        efic = round(float(ultimo['Eficiencia']), 2) if not pd.isna(ultimo['Eficiencia']) else None

        atr_series = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=OTIMIZADO_SWING['atr_period'])
        atr = float(atr_series.iloc[-1]) if not atr_series.empty and not pd.isna(atr_series.iloc[-1]) else 0.0

        padrao_altista = True
        padrao_baixista = True
        if df_d is not None:
            try:
                df_d_local = df_d.copy()
                if not df_d_local.empty:
                    if isinstance(df_d_local.columns, pd.MultiIndex):
                        df_d_local.columns = df_d_local.columns.droplevel(1)
                    df_d_local.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
                    padrao_altista = detectar_padrao_altista(df_d_local)
                    padrao_baixista = detectar_padrao_baixista(df_d_local)
            except:
                pass

        sentimento = obter_sentimento_ticker(ticker)
        volume_anormal = detectar_volume_anormal(df_w, periodo=20, limiar=OTIMIZADO_SWING['volume_limiar'])

        # Fibonacci (agora logarítmico)
        fibos = calcular_alvos_fibonacci(df_w, 'COMPRA' if padrao_altista else 'VENDA')

        setups = []

        # COMPRA
        if padrao_altista:
            stop_atr = entrada - (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            try:
                sw_low = detectar_swing_low(df_w, janela=OTIMIZADO_SWING['swing_window'])
                stop_swing = sw_low if sw_low < entrada else None
            except:
                stop_swing = None
            try:
                lta = calcular_lta_pivos(df_w, janela_pivo=OTIMIZADO_SWING['lta_pivo_window'])
                stop_lta = lta if (lta is not None and lta < entrada) else None
            except:
                stop_lta = None

            for metodo, stop_loss in [('ATR', stop_atr), ('Swing Low', stop_swing), ('LTA Pivôs', stop_lta)]:
                if stop_loss is None or stop_loss <= 0 or stop_loss >= entrada:
                    continue
                risco = entrada - stop_loss
                alvo = entrada + (risco * 3)
                if alvo <= recent_high * 1.05:
                    setups.append({
                        'Ticker': ticker,
                        'Modalidade': 'Swing',
                        'Direcao': 'COMPRA',
                        'Entrada': round(entrada, 2),
                        'Método Stop': metodo,
                        'Stop Loss': round(stop_loss, 2),
                        'Risco (R$)': round(risco, 2),
                        'Alvo 3:1': round(alvo, 2),
                        'Resistência': round(recent_high, 2),
                        'Suporte': round(recent_low, 2),
                        'Regime': reg_atual,
                        'Eficiência': efic,
                        'Sentimento': round(sentimento, 2),
                        'Volume Anormal': volume_anormal,
                        'Alvos Fibonacci': fibos,
                    })

        # VENDA
        if padrao_baixista:
            stop_atr = entrada + (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            try:
                sw_high = detectar_swing_high(df_w, janela=OTIMIZADO_SWING['swing_window'])
                stop_swing = sw_high if sw_high > entrada else None
            except:
                stop_swing = None
            try:
                ltb = calcular_ltb_pivos(df_w, janela_pivo=OTIMIZADO_SWING['ltb_pivo_window'])
                stop_ltb = ltb if (ltb is not None and ltb > entrada) else None
            except:
                stop_ltb = None

            for metodo, stop_loss in [('ATR', stop_atr), ('Swing High', stop_swing), ('LTB Pivôs', stop_ltb)]:
                if stop_loss is None or stop_loss <= entrada:
                    continue
                risco = stop_loss - entrada
                alvo = entrada - (risco * 3)
                if alvo >= recent_low * 0.95:
                    setups.append({
                        'Ticker': ticker,
                        'Modalidade': 'Swing',
                        'Direcao': 'VENDA',
                        'Entrada': round(entrada, 2),
                        'Método Stop': metodo,
                        'Stop Loss': round(stop_loss, 2),
                        'Risco (R$)': round(risco, 2),
                        'Alvo 3:1': round(alvo, 2),
                        'Resistência': round(recent_high, 2),
                        'Suporte': round(recent_low, 2),
                        'Regime': reg_atual,
                        'Eficiência': efic,
                        'Sentimento': round(sentimento, 2),
                        'Volume Anormal': volume_anormal,
                        'Alvos Fibonacci': fibos,
                    })

        return setups if setups else None
    except:
        return None

def analisar_position_trade(ticker, df_m=None):
    try:
        if df_m is None:
            return None
        df_m = df_m.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_m.columns):
            rename_map = {'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}
            df_m.rename(columns={k:v for k,v in rename_map.items() if k in df_m.columns}, inplace=True)

        df_m.sort_index(inplace=True)
        if not isinstance(df_m.index, pd.DatetimeIndex):
            df_m.index = pd.to_datetime(df_m.index)

        df_m['Eficiencia'] = calcular_eficiencia_candle(df_m)
        df_m['Regime'] = detectar_regime(df_m)

        ultimo = df_m.iloc[-1]
        entrada = float(ultimo['Close'])
        if pd.isna(entrada) or entrada <= 0:
            return None

        lookback = min(60, len(df_m))
        recent_high = float(df_m['High'].rolling(window=lookback).max().iloc[-1])
        recent_low  = float(df_m['Low'].rolling(window=lookback).min().iloc[-1])

        reg_atual = int(ultimo['Regime']) if not pd.isna(ultimo['Regime']) else -1
        efic = round(float(ultimo['Eficiencia']), 2) if not pd.isna(ultimo['Eficiencia']) else None

        atr_series = ta.atr(df_m['High'], df_m['Low'], df_m['Close'], length=OTIMIZADO_POSITION['atr_period'])
        atr = float(atr_series.iloc[-1]) if not atr_series.empty and not pd.isna(atr_series.iloc[-1]) else 0.0

        sentimento = obter_sentimento_ticker(ticker)
        volume_anormal = detectar_volume_anormal(df_m, periodo=20, limiar=OTIMIZADO_POSITION['volume_limiar'])

        # Fibonacci (logarítmico)
        fibos = calcular_alvos_fibonacci(df_m, 'COMPRA')

        setups = []

        # COMPRA
        stop_atr = entrada - (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        try:
            sw_low = detectar_swing_low(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            stop_swing = sw_low if sw_low < entrada else None
        except:
            stop_swing = None
        try:
            lta = calcular_lta_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['lta_pivo_window'])
            stop_lta = lta if (lta is not None and lta < entrada) else None
        except:
            stop_lta = None

        for metodo, stop_loss in [('ATR', stop_atr), ('Swing Low', stop_swing), ('LTA Pivôs', stop_lta)]:
            if stop_loss is None or stop_loss <= 0 or stop_loss >= entrada:
                continue
            risco = entrada - stop_loss
            alvo = entrada + (risco * 3)
            if alvo <= recent_high * 1.10:
                setups.append({
                    'Ticker': ticker,
                    'Modalidade': 'Position',
                    'Direcao': 'COMPRA',
                    'Entrada': round(entrada, 2),
                    'Método Stop': metodo,
                    'Stop Loss': round(stop_loss, 2),
                    'Risco (R$)': round(risco, 2),
                    'Alvo 3:1': round(alvo, 2),
                    'Resistência': round(recent_high, 2),
                    'Suporte': round(recent_low, 2),
                    'Regime': reg_atual,
                    'Eficiência': efic,
                    'Sentimento': round(sentimento, 2),
                    'Volume Anormal': volume_anormal,
                    'Alvos Fibonacci': fibos,
                })

        # VENDA
        stop_atr = entrada + (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        try:
            sw_high = detectar_swing_high(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            stop_swing = sw_high if sw_high > entrada else None
        except:
            stop_swing = None
        try:
            ltb = calcular_ltb_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['ltb_pivo_window'])
            stop_ltb = ltb if (ltb is not None and ltb > entrada) else None
        except:
            stop_ltb = None

        for metodo, stop_loss in [('ATR', stop_atr), ('Swing High', stop_swing), ('LTB Pivôs', stop_ltb)]:
            if stop_loss is None or stop_loss <= entrada:
                continue
            risco = stop_loss - entrada
            alvo = entrada - (risco * 3)
            if alvo >= recent_low * 0.90:
                setups.append({
                    'Ticker': ticker,
                    'Modalidade': 'Position',
                    'Direcao': 'VENDA',
                    'Entrada': round(entrada, 2),
                    'Método Stop': metodo,
                    'Stop Loss': round(stop_loss, 2),
                    'Risco (R$)': round(risco, 2),
                    'Alvo 3:1': round(alvo, 2),
                    'Resistência': round(recent_high, 2),
                    'Suporte': round(recent_low, 2),
                    'Regime': reg_atual,
                    'Eficiência': efic,
                    'Sentimento': round(sentimento, 2),
                    'Volume Anormal': volume_anormal,
                    'Alvos Fibonacci': fibos,
                })

        return setups if setups else None
    except:
        return None

print("✅ Célula 3 carregada (Swing + Position com LTA/LTB e Fibonacci Log).")


In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL (COM LOTES, RETRY, E-MAIL)
# =============================================================================

try:
    from google.colab import userdata
    if not SENHA_APP:
        SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except:
    pass

VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
PRECO_MINIMO = 5.00
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.20
EXIGIR_CONFLUENCIA = True

CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
FRACAO_KELLY = 0.25

SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA',
                      'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

def montar_tabela_html(oportunidades, titulo):
    if not oportunidades:
        return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse;'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Entrada</th><th>Stop</th><th>Alvo 3:1</th><th>Alvo 161.8%</th><th>Alvo 261.8%</th><th>Lote</th><th>Score</th><th>Wyckoff</th></tr>"
    for op in oportunidades:
        fibos = op.get('Alvos Fibonacci', {})
        fibo_161 = fibos.get('161.8%', 'N/A')
        fibo_261 = fibos.get('261.8%', 'N/A')
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op['Alvo 3:1']:.2f}</td><td>R$ {fibo_161}</td><td>R$ {fibo_261}</td><td>{op['Lote']}</td><td>{op.get('Score', 'N/A')}</td><td>{op.get('Fase Wyckoff', 'N/A')}</td></tr>"
    corpo += "</table><br>"
    return corpo

def enviar_email_ou_exibir(oportunidades, modalidade):
    if not oportunidades:
        print(f"ℹ️ Nenhuma oportunidade de {modalidade} encontrada.")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} 3:1 - {datetime.now().strftime('%d/%m/%Y')}"
            corpo = f"<h2>Setups {modalidade} Detectados (Score de Confiança, Wyckoff e Fibonacci)</h2>"
            corpo += montar_tabela_html(oportunidades, "Setups Aprovados")
            msg.attach(MIMEText(corpo, 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            print(f"✅ E-mail ({modalidade}) enviado para {EMAIL_REMETENTE}")
        except Exception as e:
            print(f"❌ Falha no e-mail ({modalidade}): {e}")
    else:
        print(f"📧 E-mail não configurado. Exibindo {modalidade} na tela.\n")

    df_op = pd.DataFrame(oportunidades)
    colunas = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)',
               'Alvo 3:1', 'Resistência', 'Suporte', 'Regime', 'Eficiência',
               'Sentimento', 'Volume Anormal', 'Fase Wyckoff', 'Score', 'Kelly %', 'Lote']
    try:
        from IPython.display import display
        display(df_op[colunas].sort_values(['Direcao', 'Ticker']))
    except:
        print(df_op[colunas].sort_values(['Direcao', 'Ticker']).to_string())

    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[colunas].to_csv(csv_name, index=False)
    try:
        from google.colab import files
        files.download(csv_name)
        print(f"\n📁 Arquivo '{csv_name}' baixado.")
    except ImportError:
        print(f"\n📁 Arquivo '{csv_name}' salvo (GitHub Actions não faz download automático).")

def obter_tickers_b3():
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cols = row.find_all('td')
            if cols:
                t = cols[0].text.strip()
                if t and not t.startswith('#'):
                    tickers.append(t)
        if tickers:
            return tickers
    except:
        pass
    return ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4']

print("🔍 Obtendo tickers...")
tickers_b3 = obter_tickers_b3()
print(f"✅ {len(tickers_b3)} tickers.")

# -----------------------------------------------------------------------------
# FILTRO DE LIQUIDEZ EM LOTES
# -----------------------------------------------------------------------------
tickers_yahoo = [t + ".SA" for t in tickers_b3]
print("📦 Baixando dados diários em lotes para filtro de liquidez...")
tickers_liquidos = []
BATCH_SIZE = 50
for i in range(0, len(tickers_yahoo), BATCH_SIZE):
    batch = tickers_yahoo[i:i+BATCH_SIZE]
    try:
        data_batch = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS:
                continue
            try:
                if t not in data_batch:
                    continue
                df_t = data_batch[t].copy()
                if isinstance(df_t.columns, pd.MultiIndex):
                    df_t.columns = df_t.columns.droplevel(1)
                df_t.columns = [col.lower() for col in df_t.columns]
                rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
                df_t.rename(columns={k:v for k,v in rename_map.items() if k in df_t.columns}, inplace=True)
                if 'Volume' not in df_t.columns or df_t.empty:
                    continue
                vol_medio = df_t['Volume'].rolling(21).mean().iloc[-1]
                preco = df_t['Close'].iloc[-1]
                if pd.notna(vol_medio) and pd.notna(preco):
                    if vol_medio >= VOLUME_MINIMO_ACAO and (vol_medio * preco) >= VOLUME_FINANCEIRO_MINIMO:
                        tickers_liquidos.append(t)
            except:
                continue
    except Exception as e:
        print(f"⚠️ Erro no lote {i//BATCH_SIZE}: {e}")
    time.sleep(2)

if len(tickers_liquidos) < 10:
    print("⚠️ Poucos ativos líquidos. Usando fallback.")
    tickers_liquidos = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'BBDC4.SA', 'BBAS3.SA', 'ABEV3.SA', 'WEGE3.SA', 'RADL3.SA', 'SUZB3.SA', 'GGBR4.SA']

print(f"💧 {len(tickers_liquidos)} ativos líquidos.")

print("📦 Baixando dados semanais em lote...")
data_w = yf.download(tickers_liquidos, period='2y', interval='1wk', group_by='ticker', progress=False, auto_adjust=True)

print("📦 Baixando dados diários (confirmação) em lote...")
data_d = yf.download(tickers_liquidos, period='1y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)

print("📦 Baixando dados mensais em lote...")
data_m = yf.download(tickers_liquidos, period='max', interval='1mo', group_by='ticker', progress=False, auto_adjust=True)

# -----------------------------------------------------------------------------
# CACHE MACRO
# -----------------------------------------------------------------------------
cache_macro = {}
for ticker_ref, (benchmark, _) in MACRO_REFERENCE.items():
    if benchmark not in cache_macro:
        try:
            df_bench = yf.download(benchmark, period='1y', interval='1wk', progress=False, auto_adjust=True)
            if not df_bench.empty:
                cache_macro[benchmark] = df_bench['Close'].values
        except:
            cache_macro[benchmark] = None

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, FRACAO_KELLY)
risco_maximo = CAPITAL_TOTAL * kelly_pct

oportunidades_swing = []
oportunidades_position = []

for i, ticker in enumerate(tickers_liquidos):
    print(f"Analisando {ticker} ({i+1}/{len(tickers_liquidos)})...", end='\r')

    def get_df(data, ticker):
        if data is not None and ticker in data:
            df = data[ticker].copy()
            df.columns = [col.lower() for col in df.columns]
            df.rename(columns={'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}, inplace=True)
            return df
        return None

    df_w = get_df(data_w, ticker)
    df_d = get_df(data_d, ticker)
    df_m = get_df(data_m, ticker)

    # Volume financeiro médio (proxy de liquidez)
    vol_fin_medio = None
    if df_w is not None and not df_w.empty:
        vol_fin_medio = (df_w['Volume'] * df_w['Close']).rolling(20).mean().iloc[-1]

    # --- SWING TRADE ---
    if df_w is not None and not df_w.empty:
        res_swing = analisar_swing_trade(ticker, df_w=df_w, df_d=df_d)
        if res_swing:
            for r in res_swing:
                e = r['Entrada']
                risco_pct = r['Risco (R$)'] / e
                if e < PRECO_MINIMO: continue
                if risco_pct < RISCO_PERCENTUAL_MINIMO: continue
                if risco_pct > RISCO_PERCENTUAL_MAXIMO: continue
                if EXIGIR_CONFLUENCIA:
                    if r['Regime'] not in [1,2]: continue
                    if r['Eficiência'] is None or r['Eficiência'] < 0.6: continue
                if OTIMIZADO_SWING['mm200_semanal']:
                    mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
                    if pd.isna(mm200w): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm200w) or (r['Direcao'] == 'VENDA' and e > mm200w):
                        continue
                if OTIMIZADO_SWING['mm200_diaria'] and df_d is not None:
                    mm200d = df_d['Close'].rolling(200).mean().iloc[-1]
                    if pd.isna(mm200d): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm200d) or (r['Direcao'] == 'VENDA' and e > mm200d):
                        continue

                # CAMADA 1: ALINHAMENTO MACRO
                alinhado_macro, score_macro = verificar_alinhamento_macro(ticker, r['Direcao'], cache_macro)
                if not alinhado_macro:
                    continue

                # CAMADA 2: QUALIDADE DO VOLUME
                qualidade_volume, score_volume = avaliar_qualidade_volume(df_w, r['Direcao'])

                # FASE WYCKOFF
                fase_wyckoff = detectar_fase_wyckoff(df_w, suporte=r.get('Suporte'), resistencia=r.get('Resistência'))
                if (r['Direcao'] == 'COMPRA' and fase_wyckoff == 'DISTRIBUICAO') or \
                   (r['Direcao'] == 'VENDA' and fase_wyckoff == 'ACUMULACAO'):
                    continue

                # CAMADA 3: SCORE DE CONFIANÇA E KELLY AJUSTADO POR LIQUIDEZ
                multiplicador_confianca = calcular_score_confianca(
                    r, alinhado_macro, qualidade_volume, score_volume, fase_wyckoff
                )

                # Fator de liquidez (proteção contra slippage)
                if pd.isna(vol_fin_medio):
                    fator_liquidez = 0.5
                else:
                    fator_liquidez = min(1.0, vol_fin_medio / LIMITE_LIQUIDEZ_FINANCEIRA)

                lote_base = int(risco_maximo / r['Risco (R$)'])
                lote_ajustado = int(lote_base * multiplicador_confianca * fator_liquidez)
                if lote_ajustado == 0: continue

                r['Kelly %'] = round((kelly_pct * multiplicador_confianca * fator_liquidez) * 100, 2)
                r['Lote'] = lote_ajustado
                r['Score'] = int(multiplicador_confianca * 100)
                r['Alinham. Macro'] = 'Sim' if alinhado_macro else 'Não'
                r['Qualid. Volume'] = qualidade_volume
                r['Fase Wyckoff'] = fase_wyckoff
                oportunidades_swing.append(r)

    # --- POSITION TRADE ---
    if df_m is not None and not df_m.empty:
        res_pos = analisar_position_trade(ticker, df_m=df_m)
        if res_pos:
            for r in res_pos:
                e = r['Entrada']
                risco_pct = r['Risco (R$)'] / e
                if e < PRECO_MINIMO: continue
                if risco_pct < 0.03: continue
                if risco_pct > 0.30: continue
                if EXIGIR_CONFLUENCIA:
                    if r['Regime'] not in [1,2]: continue
                    if r['Eficiência'] is None or r['Eficiência'] < 0.5: continue
                if OTIMIZADO_POSITION['mm50_mensal']:
                    mm50m = df_m['Close'].rolling(50).mean().iloc[-1]
                    if pd.isna(mm50m): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm50m) or (r['Direcao'] == 'VENDA' and e > mm50m):
                        continue

                # Camadas também para Position
                alinhado_macro, score_macro = verificar_alinhamento_macro(ticker, r['Direcao'], cache_macro)
                if not alinhado_macro:
                    continue
                qualidade_volume, score_volume = avaliar_qualidade_volume(df_m, r['Direcao'])
                fase_wyckoff = detectar_fase_wyckoff(df_m, suporte=r.get('Suporte'), resistencia=r.get('Resistência'))
                if (r['Direcao'] == 'COMPRA' and fase_wyckoff == 'DISTRIBUICAO') or \
                   (r['Direcao'] == 'VENDA' and fase_wyckoff == 'ACUMULACAO'):
                    continue
                multiplicador_confianca = calcular_score_confianca(
                    r, alinhado_macro, qualidade_volume, score_volume, fase_wyckoff
                )

                vol_fin_medio_m = (df_m['Volume'] * df_m['Close']).rolling(20).mean().iloc[-1]
                if pd.isna(vol_fin_medio_m):
                    fator_liquidez = 0.5
                else:
                    fator_liquidez = min(1.0, vol_fin_medio_m / LIMITE_LIQUIDEZ_FINANCEIRA)

                lote_base = int(risco_maximo / r['Risco (R$)'])
                lote_ajustado = int(lote_base * multiplicador_confianca * fator_liquidez)
                if lote_ajustado == 0: continue

                r['Kelly %'] = round((kelly_pct * multiplicador_confianca * fator_liquidez) * 100, 2)
                r['Lote'] = lote_ajustado
                r['Score'] = int(multiplicador_confianca * 100)
                r['Alinham. Macro'] = 'Sim' if alinhado_macro else 'Não'
                r['Qualid. Volume'] = qualidade_volume
                r['Fase Wyckoff'] = fase_wyckoff
                oportunidades_position.append(r)

print(f"\n🎯 Swing Trade: {len(oportunidades_swing)} setups | Position Trade: {len(oportunidades_position)} setups")
print(f"   Kelly recomendado: {kelly_pct*100:.2f}% do capital (R$ {risco_maximo:.2f})")

if oportunidades_swing:
    enviar_email_ou_exibir(oportunidades_swing, "Swing Trade")
else:
    print("\n⚪ Nenhum setup de Swing Trade hoje.")

if oportunidades_position:
    enviar_email_ou_exibir(oportunidades_position, "Position Trade")
else:
    print("\n⚪ Nenhum setup de Position Trade hoje.")
